# Experiment 1: compare approaches
This notebook reads only completed portable analysis summaries and histogram CSV files. It never opens raw recordings, models, or simulator code. Loading ranges and motion selections are compared separately.

In [ ]:
import os, json, sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Find this checkout whether Jupyter starts in the repository or notebooks directory.
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebooks" / "exp1_analysis.py").exists())
sys.path.insert(0, str(REPO / "notebooks"))
import exp1_analysis as exp1
# Refresh helper changes even when this Jupyter kernel was already running.
import importlib
exp1 = importlib.reload(exp1)

CONFIG = {
    "summary_roots": [str(REPO / "analysis/exp1")],
    "output_dir": str(REPO / "analysis/exp1_comparison/in"),
    "method_order": [
        "b1z1_unifp_original", "b1z1_ppo_pos",
        "b1z1_pact_ab4_coupled_none", "b1z1_pact_ab5_coupled_concat",
        "b1z1_pact_ab6_position_film", "b1z1_pact_ab7_position_full",
        "b1z1_pact_ab8_coupled_film", "b1z1_pact_ab9_representation",
        "b1z1_pact_ab10_actor", "b1z1_pact"],
    "display_names": {"b1z1_pact": "PILOT", "b1z1_unifp_original": "UniFP", "b1z1_ppo_pos": "Pos"},
    "focused_methods": ["b1z1_pact", "b1z1_unifp_original", "b1z1_ppo_pos"],
    "load_range": "in",  # Analyze "out" separately.
    "motion": "combined",  # "stationary", "locomotion", or explicitly exported "combined".
    "motion_modes_table": True,  # Separate stationary/locomotion table; does not change motion above.
    "motion_modes_condition": "combined-disturbance",  # Uses load_range and focused_methods above.
    "response_table": True,
    "optional_metrics": [],  # e.g. ["torso_angular_acceleration_rms", "ee_linear_acceleration_rms"]
    "additional_plots": ["step", "ecdf", "box"],  # Set [] to export only the original histogram.
}
if os.environ.get("EXP1_COMPARE_CONFIG"):
    CONFIG.update(json.loads(Path(os.environ["EXP1_COMPARE_CONFIG"]).read_text()))
CONFIG


## Validation and tables
The disturbance-response table also reports **EE recovered [%]** (EE-position tolerance only) and **Velocity recovered [%]** (planar velocity and yaw-rate tolerances). Each uses the same eligible events, response window, contiguous dwell and whole-trial failure censoring as joint recovery. Rerun summarization for all approaches to populate these new metrics; comparison never estimates them from joint recovery.
Protocols, units, histogram bins/frames and seed coverage must agree. Duplicate method summaries are rejected. Missing results display as an em dash. Each cell reports **mean ± sample SD across evaluation seeds**; tracking/failure/recovery fractions are displayed as percentages.

CSV and LaTeX tables are exported. The main table uses text width; focused tables use column width (`graphicx` and `booktabs` required). Times New Roman is used for figures if installed, with a serif fallback.

Immediately after the main table, `motion_modes_table` compares stationary manipulation and locomotion with manipulation for the focused methods: EE position error [m], planar velocity error [m/s], yaw-rate error [rad/s], joint S_track [%], velocity-only S_track [%], manipulation-only S_track [%], and failure rate [%]. Velocity-only requires both planar velocity and yaw-rate tolerances; manipulation-only requires the EE-position tolerance. All scores use the full prescribed horizon with zero credit after failure. Rerun the summarization notebook for every approach to calculate the new scores from transitions; they cannot be inferred from joint S_track. This comparison reads only portable-summary strata for `motion_modes_condition`, without changing global motion selection. `motion_modes.csv` retains numeric means, sample SDs and counts (fractions converted once to percent); `motion_modes.tex` is a single-column grouped table. Missing results are never replaced with combined-motion results.

In [ ]:
tables, figure, diagnostics = exp1.compare(CONFIG)
for name, table in tables.items():
    print(name)
    display(table)
print("\n".join(diagnostics) or "Compatible complete comparison.")


## Signed EE-error distributions
Overlaid histogram bars show equal-seed mean percentage of samples per bin; error bars show sample SD across seeds. These are bin probabilities, not density curves. Rows are x/y/z; columns are nominal and combined disturbances. All methods share bins and axes. Tails are preserved in the histogram exports and `ee_distribution_data.csv`; probabilities are **not** renormalized to hide tails. Vector PDF and PNG are exported alongside tables.

In [ ]:
display(figure)
# ECDF is binned; exact equal-seed box statistics require rerunning the summarization notebook.
# Boxes show median and 25-75% spread; whiskers are 5-95%, not Tukey fences or confidence intervals.
from IPython.display import Image
for kind in CONFIG.get("additional_plots", ["step", "ecdf", "box"]):
    print(kind)
    display(Image(filename=str(Path(CONFIG["output_dir"]) / f"ee_{kind}.png")))
print("Exported:", CONFIG["output_dir"])
